In [0]:
## read raw data into dataframe
customers = spark.read.table("db_academy.pii_data.customers_raw")

display(customers.limit(10))

In [0]:
from pyspark.sql.functions import sha1, sha2, md5, col

customers_hashing = (
    spark.read.table("db_academy.pii_data.customers_raw")
    .select(
        ### customer_id is not PII
        "customer_id",

        ### full_name is PII, using sha2-256
        sha2("full_name", 256).alias("full_name"),

        ### email is PII, using md5
        md5("email").alias("email"),

        ### phone is PII, using sha1, cast the int to string
        sha1(col("phone").cast("string")).alias("phone"),

        ### ssn is PII, using sha2-512
        sha2("ssn", 512).alias("ssn"),

        ### date_of_birth is PII, using sha2-384
        sha2(col("date_of_birth").cast("string"), 384).alias("date_of_birth"),

        ### address is PII, using sha2-224
        sha2("address", 224).alias("address"),

        ### city is PII, using md5
        md5("city").alias("city"),

        ### country is not PII
        "country"
    )
)

display(customers_hashing.limit(5))



In [0]:
## write hashing table
customers_hashing.write.mode("overwrite").saveAsTable("db_academy.pii_data.customers_hashed")

In [0]:
%sql

SELECT * FROM db_academy.pii_data.customers_hashed

In [0]:
%sql

SELECT 'email' as col, *
FROM db_academy.pii_data.customers_hashed
WHERE email = 'lauraanderson@example.org'
UNION
SELECT 'customer_id' as col, *
FROM db_academy.pii_data.customers_hashed
WHERE customer_id = 1

In [0]:
%sql

SELECT *
FROM db_academy.pii_data.customers_raw
WHERE email = 'lauraanderson@example.org'